In [1]:
import MetaTrader5 as mt
from datetime import datetime
import time

# =============================================================================
# MT5 INIT
# =============================================================================
if not mt.initialize():
    print("MT5 init failed")
    quit()

login  = 433031713
password = "Soham@987"
server   = "Exness-MT5Trial7"

if not mt.login(login, password=password, server=server):
    print("Login failed:", mt.last_error())
    quit()

print("Connected to MT5")

Connected to MT5


In [19]:
TICKER = "XAUUSDm"

BASE_LOT = 1.0
MULTIPLIER = 2.0

GRID_POINTS_DISTANCE = 1.0          # TOTAL GRID
HALF_GRID_DISTANCE = GRID_POINTS_DISTANCE / 2.0

TP_MULTIPLIER = 4.0
SL_DIFFERENCE = 0.1
MAX_LEVELS = 4
MAGIC_NUMBER = 1234567

CMP = mt.symbol_info_tick(TICKER)
P0 = round((CMP.bid + CMP.ask) / 2, 4)
ASK = CMP.ask
BID = CMP.bid

ABOVE_GRID = P0 + (HALF_GRID_DISTANCE)
BELOW_GRID = P0 - (HALF_GRID_DISTANCE)

BUY_SL = BELOW_GRID - SL_DIFFERENCE
SELL_SL = ABOVE_GRID + SL_DIFFERENCE
BUY_TP = ABOVE_GRID + (GRID_POINTS_DISTANCE * TP_MULTIPLIER)
SELL_TP = BELOW_GRID - (GRID_POINTS_DISTANCE * TP_MULTIPLIER)

In [7]:
print(P0, ASK, BID)
print(BELOW_GRID)

5336.826 5336.906 5336.746
5336.326


In [25]:
TICKER = "XAUUSDm"

BASE_LOT = 1.0
MULTIPLIER = 2.0

GRID_POINTS_DISTANCE = 1.0          # TOTAL GRID
HALF_GRID_DISTANCE = GRID_POINTS_DISTANCE / 2.0

TP_MULTIPLIER = 4.0
SL_DIFFERENCE = 0.1
MAX_LEVELS = 4
MAGIC_NUMBER = 1234567

CMP = mt.symbol_info_tick(TICKER)
P0 = round((CMP.bid + CMP.ask) / 2, 4)
ASK = CMP.ask
BID = CMP.bid

ABOVE_GRID = P0 + (HALF_GRID_DISTANCE)
BELOW_GRID = P0 - (HALF_GRID_DISTANCE)

BUY_SL = BELOW_GRID - SL_DIFFERENCE
SELL_SL = ABOVE_GRID + SL_DIFFERENCE
BUY_TP = ABOVE_GRID + (GRID_POINTS_DISTANCE * TP_MULTIPLIER)
SELL_TP = BELOW_GRID - (GRID_POINTS_DISTANCE * TP_MULTIPLIER)

def place_order(type, lot, price, comment):
    req = {
        "action": mt.TRADE_ACTION_PENDING,
        "symbol": TICKER,
        "volume": lot,
        "type": type,
        "price": price,
        "magic": MAGIC_NUMBER,
        "comment": comment,
        "type_time": mt.ORDER_TIME_GTC,
        "type_filling": mt.ORDER_FILLING_RETURN
    }
    result = mt.order_send(req)
    print(f"{comment} → retcode={result.retcode}")
    return result  

place_order(mt.ORDER_TYPE_SELL_STOP, BASE_LOT, BELOW_GRID, "comment")

comment → retcode=10009


OrderSendResult(retcode=10009, deal=0, order=3309101658, volume=1.0, price=0.0, bid=5335.212, ask=5335.372, comment='comment', request_id=548750962, retcode_external=0, request=TradeRequest(action=5, magic=1234567, order=0, symbol='XAUUSDm', volume=1.0, price=5334.102, stoplimit=0.0, sl=0.0, tp=0.0, deviation=0, type=5, type_filling=2, type_time=0, expiration=0, comment='comment', position=0, position_by=0))

In [22]:
# Remove Sell Order
orders = mt.orders_get(symbol=TICKER)
sell_ticket = next((o.ticket for o in orders if o.type == mt.ORDER_TYPE_SELL_STOP), None)
if sell_ticket:
    mt.order_send({
        "action": mt.TRADE_ACTION_REMOVE,
        "order": sell_ticket
    })